In [ ]:
!pip3 install git+https://github.com/NetManAIOps/sktime.git

In [ ]:
# Time Series Forecasting with sktime Seq2Seq-style forecasters
# This notebook avoids direct torch model code and uses sktime APIs only
# Models: LTSFLinear / LTSFDLinear

In [ ]:
# Optional dependency install
# !pip install -U sktime torch

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sktime.datasets import load_airline
from sktime.forecasting.base import ForecastingHorizon
from sktime.forecasting.ltsf import LTSFLinearForecaster, LTSFDLinearForecaster
from sktime.performance_metrics.forecasting import mean_absolute_error, mean_absolute_percentage_error

In [ ]:
print("Loading data...")
airline = load_airline()
print("Dataset shape:", airline.shape)
print(airline.head())
airline.tail()

In [ ]:
plt.figure(figsize=(12, 6))
airline.plot()
plt.title('Airline Passenger Data')
plt.xlabel('Time')
plt.ylabel('Passengers')
plt.grid(True)
plt.show()

In [ ]:
split_point = int(len(airline) * 0.8)
y_train, y_test = airline.iloc[:split_point], airline.iloc[split_point:]

In [ ]:
fh = ForecastingHorizon(y_test.index, is_relative=False)
pred_len = len(y_test)
seq_len = 24
print(f"Train={len(y_train)}, Test={len(y_test)}, seq_len={seq_len}, pred_len={pred_len}")

In [ ]:
# LTSFLinear as baseline seq2seq-style global forecaster
linear_fcst = LTSFLinearForecaster(
    seq_len=seq_len,
    pred_len=pred_len,
    num_epochs=20,
    batch_size=8,
    lr=0.001
)

In [ ]:
print("Training LTSFLinear...")
linear_fcst.fit(y_train, fh=fh)
y_pred_linear = linear_fcst.predict(fh)

mae_linear = mean_absolute_error(y_test, y_pred_linear)
mape_linear = mean_absolute_percentage_error(y_test, y_pred_linear)
print(f"LTSFLinear MAE: {mae_linear:.2f}, MAPE: {mape_linear:.2%}")

In [ ]:
# LTSFDLinear for decomposition-style variant
dlinear_fcst = LTSFDLinearForecaster(
    seq_len=seq_len,
    pred_len=pred_len,
    num_epochs=20,
    batch_size=8,
    lr=0.001
)

In [ ]:
print("Training LTSFDLinear...")
dlinear_fcst.fit(y_train, fh=fh)
y_pred_dlinear = dlinear_fcst.predict(fh)

mae_dlinear = mean_absolute_error(y_test, y_pred_dlinear)
mape_dlinear = mean_absolute_percentage_error(y_test, y_pred_dlinear)
print(f"LTSFDLinear MAE: {mae_dlinear:.2f}, MAPE: {mape_dlinear:.2%}")

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(y_train.index, y_train.values, label="Train")
plt.plot(y_test.index, y_test.values, label="Test")
plt.plot(y_pred_linear.index, y_pred_linear.values, label="LTSFLinear")
plt.plot(y_pred_dlinear.index, y_pred_dlinear.values, label="LTSFDLinear")
plt.title("Seq2Seq-style Forecasting with sktime")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
future_steps = 12
future_fh = np.arange(1, future_steps + 1)
future_pred_linear = linear_fcst.predict(future_fh)
future_pred_dlinear = dlinear_fcst.predict(future_fh)

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(airline.index, airline.values, label="Historical")
plt.plot(future_pred_linear.index, future_pred_linear.values, "r--", label="Future LTSFLinear")
plt.plot(future_pred_dlinear.index, future_pred_dlinear.values, "g--", label="Future LTSFDLinear")
plt.title("Future Forecasts (next 12 steps)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
print("Summary:")
print(f"LTSFLinear   -> MAE={mae_linear:.2f}, MAPE={mape_linear:.2%}")
print(f"LTSFDLinear  -> MAE={mae_dlinear:.2f}, MAPE={mape_dlinear:.2%}")